In [1]:
import torch
import numpy as np
import cv2
from PIL import Image
from transformers import AutoModel, AutoProcessor, AutoTokenizer
import sys
sys.path.append("/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/playground/MedCLIP-SAMv2/saliency_maps")
from scripts.methods import vision_heatmap_iba, text_heatmap_iba

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load MedCLIP-SAMv2 (BiomedCLIP)
model = AutoModel.from_pretrained("chuhac/BiomedCLIP-vit-bert-hf").to(device)
processor = AutoProcessor.from_pretrained("chuhac/BiomedCLIP-vit-bert-hf")
tokenizer = AutoTokenizer.from_pretrained("chuhac/BiomedCLIP-vit-bert-hf")

def compute_text_prompted_segmentation_score(model, image_path, text, vbeta=0.1, vvar=1, vlayer=9, tbeta=0.1, tvar=1, tlayer=9):
    """
    Compute a score for how well the segmentation aligns with the text prompt.

    Args:
        model: MedCLIP-SAMv2 model
        image_path: Path to the input image
        text: Text prompt describing the region of interest
        vbeta, vvar, vlayer: Parameters for vision saliency mapping
        tbeta, tvar, tlayer: Parameters for text saliency mapping

    Returns:
        segmentation_score: Score measuring text-image alignment in segmentation
    """

    # Load and preprocess the image
    image = Image.open(image_path).convert('RGB')
    image_feat = processor(images=image, return_tensors="pt")['pixel_values'].to(device)

    # Tokenize text
    text_ids = torch.tensor([tokenizer.encode(text, add_special_tokens=True)]).to(device)
    
    # Generate Saliency Maps
    print("Generating saliency map for vision...")
    vmap = vision_heatmap_iba(text_ids, image_feat, model, vlayer, vbeta, vvar)

    print("Generating saliency map for text...")
    tmap = text_heatmap_iba(text_ids, image_feat, model, tlayer, tbeta, tvar)

    # Compute Confidence Score from Saliency Map
    def compute_saliency_score(saliency_map):
        """Compute the mean intensity of the saliency map as a confidence score."""
        saliency_map = np.array(saliency_map)
        saliency_map = cv2.resize(saliency_map, (224, 224), interpolation=cv2.INTER_NEAREST)  # Normalize size
        score = np.mean(saliency_map)  # Average saliency intensity
        return score

    vision_score = compute_saliency_score(vmap)
    text_score = compute_saliency_score(tmap)

    # Normalize scores (scaling between 0-1)
    vision_score = min(1.0, vision_score / 255)
    text_score = min(1.0, text_score / 255)

    # Compute Final Segmentation Score
    segmentation_score = 0.5 * vision_score + 0.5 * text_score

    return segmentation_score

# Example Usage
image_path = "/staging/users/tpadhi1/Multimodal-Uncertainty-Quantification/playground/MedCLIP-SAMv2/assets/example.png"
text = "A medical brain MRI scan showing a well-circumscribed, extra-axial mass suggestive of a meningioma tumor."

segmentation_score = compute_text_prompted_segmentation_score(model, image_path, text)
print(f"Text-Prompted Segmentation Score: {segmentation_score:.4f}")

/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:482: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:339: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/tpadhi1/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/utils/generic.py:339: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.registe

config.json:   0%|          | 0.00/2.86k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/784M [00:00<?, ?B/s]

Some weights of CLIPModel were not initialized from the model checkpoint at chuhac/BiomedCLIP-vit-bert-hf and are newly initialized: ['text_model.final_layer_norm.bias', 'text_model.final_layer_norm.weight', 'text_projection.weight', 'vision_model.pre_layrnorm.bias', 'vision_model.pre_layrnorm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/679k [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'CLIPTokenizerFast'.


ValueError: The `backend_tokenizer` provided does not match the expected format. The CLIP tokenizer has been heavily modified from transformers version 4.17.0. You need to convert the tokenizer you are using to be compatible with this version.The easiest way to do so is `CLIPTokenizerFast.from_pretrained("path_to_local_folder_or_hub_repo, from_slow=True)`. If you want to use your existing tokenizer, you will have to revert to a version prior to 4.17.0 of transformers.